# 📉 Customer Churn Prediction Dashboard
**Author: Shreya Wargantiwar**  
**Tools: Python, Pandas, Scikit-learn, Matplotlib, Seaborn**

This notebook analyzes 7,043 telecom customer records to predict churn using Random Forest and identify high-risk customer segments.

## 📂 Step 1: Upload Your CSV File

In [ ]:
from google.colab import files
uploaded = files.upload()
print('✅ File uploaded!')

## 📦 Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
print('✅ All libraries imported!')

## 🔍 Step 3: Load & Clean Data

In [ ]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Fix TotalCharges (some are empty strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Convert Churn to binary
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)

print(f'✅ Dataset loaded: {df.shape[0]:,} customers, {df.shape[1]} columns')
print(f'\nChurn breakdown:')
print(df['Churn'].value_counts())
churn_rate = df['Churn_Binary'].mean() * 100
print(f'\n⚠️  Overall Churn Rate: {churn_rate:.1f}%')
df.head()

## 📊 Step 4: Exploratory Data Analysis

In [ ]:
print('='*50)
print('KEY STATISTICS')
print('='*50)
print(f'Total Customers     : {len(df):,}')
print(f'Churned Customers   : {df["Churn_Binary"].sum():,} ({df["Churn_Binary"].mean()*100:.1f}%)')
print(f'Avg Monthly Charges : ${df["MonthlyCharges"].mean():.2f}')
print(f'Avg Tenure          : {df["tenure"].mean():.1f} months')
print(f'\nChurn by Contract Type:')
print(df.groupby('Contract')['Churn_Binary'].mean().mul(100).round(1).sort_values(ascending=False))
print(f'\nChurn by Internet Service:')
print(df.groupby('InternetService')['Churn_Binary'].mean().mul(100).round(1).sort_values(ascending=False))

## 📈 Step 5: Visualizations Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('📉 Customer Churn Analysis Dashboard', fontsize=16, fontweight='bold')

# Plot 1: Churn Pie Chart
churn_counts = df['Churn'].value_counts()
axes[0,0].pie(churn_counts.values, labels=churn_counts.index,
              autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'], startangle=90)
axes[0,0].set_title('Churn vs Retained Customers')

# Plot 2: Monthly Charges by Churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0,1],
           boxprops=dict(color='#3498db'), medianprops=dict(color='red'))
axes[0,1].set_title('Monthly Charges by Churn Status')
axes[0,1].set_xlabel('Churn')
plt.sca(axes[0,1])
plt.title('Monthly Charges by Churn Status')

# Plot 3: Contract Type vs Churn
contract_churn = df.groupby('Contract')['Churn_Binary'].mean().mul(100).sort_values(ascending=False)
contract_churn.plot(kind='bar', ax=axes[0,2], color='#9b59b6', rot=15)
axes[0,2].set_title('Churn Rate by Contract Type (%)')
axes[0,2].set_ylabel('Churn Rate (%)')

# Plot 4: Tenure Distribution
axes[1,0].hist(df[df['Churn']=='No']['tenure'], alpha=0.7, label='Retained', color='#2ecc71', bins=30)
axes[1,0].hist(df[df['Churn']=='Yes']['tenure'], alpha=0.7, label='Churned', color='#e74c3c', bins=30)
axes[1,0].set_title('Tenure Distribution')
axes[1,0].set_xlabel('Months')
axes[1,0].legend()

# Plot 5: Internet Service vs Churn
internet_churn = df.groupby('InternetService')['Churn_Binary'].mean().mul(100).sort_values(ascending=False)
internet_churn.plot(kind='bar', ax=axes[1,1], color='#e67e22', rot=15)
axes[1,1].set_title('Churn Rate by Internet Service (%)')

# Plot 6: Payment Method vs Churn
payment_churn = df.groupby('PaymentMethod')['Churn_Binary'].mean().mul(100).sort_values(ascending=False)
payment_churn.plot(kind='barh', ax=axes[1,2], color='#1abc9c')
axes[1,2].set_title('Churn Rate by Payment Method (%)')

plt.tight_layout()
plt.savefig('churn_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard created!')

## 🤖 Step 6: Random Forest — Predict Churn

In [ ]:
# Encode categorical columns
df_ml = df.copy()
le = LabelEncoder()
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
            'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
            'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
            'MonthlyCharges', 'TotalCharges']

X = df_ml[features]
y = df_ml['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f'✅ Model Accuracy: {accuracy:.1f}%')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

## 🔑 Step 7: Feature Importance — What Causes Churn?

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
importances.head(10).plot(kind='bar', color='#e74c3c')
plt.title('Top 10 Factors That Cause Customer Churn', fontsize=13)
plt.ylabel('Importance Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('churn_feature_importance.png', dpi=150)
plt.show()

print('\nTop 5 churn factors:')
for feat, imp in importances.head(5).items():
    print(f'  {feat}: {imp:.3f}')

## 🎯 Step 8: Identify High-Risk Customers

In [ ]:
# Add churn probability to original dataframe
churn_probs = model.predict_proba(X)[:, 1]
df['Churn_Probability'] = churn_probs
df['Risk_Segment'] = pd.cut(churn_probs,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk'])

print('Customer Risk Segments:')
print(df['Risk_Segment'].value_counts())
print(f'\n🔴 High Risk Customers: {(df["Risk_Segment"]=="High Risk").sum():,}')
print(f'🟡 Medium Risk Customers: {(df["Risk_Segment"]=="Medium Risk").sum():,}')
print(f'🟢 Low Risk Customers: {(df["Risk_Segment"]=="Low Risk").sum():,}')

# Show top 10 highest risk customers
print(f'\nTop 10 Highest Risk Customers:')
high_risk = df[['customerID', 'Contract', 'MonthlyCharges', 'tenure', 'Churn_Probability']]\
    .sort_values('Churn_Probability', ascending=False).head(10)
print(high_risk.to_string(index=False))

## 💾 Step 9: Download Charts

In [ ]:
files.download('churn_dashboard.png')
files.download('churn_feature_importance.png')
print('✅ Charts downloaded!')

## 📊 Final Summary

| Metric | Value |
|---|---|
| Total Customers | 7,043 |
| Churn Rate | 26.5% |
| ML Model Accuracy | ~80% |
| Top Churn Factor | Tenure + Monthly Charges |
| High Risk Customers | Identified & Segmented |

**Author: Shreya Wargantiwar | MCA Data Science | Sri Balaji University, Pune**